# 14 — Demographics vs. Homeless Rate (County/CoC)

Correlations between homeless rate and racial demographics and veteran population at CoC level.

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

ROOT = Path().resolve().parent
df = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_county_data.csv')
df_demo = df.dropna(subset=['homeless_rate_per_10k', 'pct_black']) if 'pct_black' in df.columns else pd.DataFrame()
print(f'CoCs with demographics: {len(df_demo)} (of {len(df)} total)')

CoCs with demographics: 0 (of 102 total)


In [2]:
if len(df_demo) > 5:
    slope, intercept, r, p, se = stats.linregress(df_demo['pct_black'], df_demo['homeless_rate_per_10k'])
    r2 = r ** 2
    print(f'% Black vs rate: r={r:.3f}, R²={r2:.3f}, p={p:.4f}')
    x_range = [df_demo['pct_black'].min(), df_demo['pct_black'].max()]
    y_range = [slope * x + intercept for x in x_range]
    fig = px.scatter(
        df_demo, x='pct_black', y='homeless_rate_per_10k',
        color='homeless_rate_per_10k', color_continuous_scale='Reds',
        hover_name='coc_name',
        title=f'% Black Population vs. Homeless Rate (CoC Level)<br><sup>r={r:.3f}, R²={r2:.3f}, p={p:.4f}</sup>',
        labels={'pct_black': '% Black (non-Hispanic)', 'homeless_rate_per_10k': 'Rate per 10k'},
    )
    fig.add_trace(go.Scatter(x=x_range, y=y_range, mode='lines', name='Regression', line=dict(color='darkred', dash='dash')))
    fig.show()
else:
    print('Census demographic data not available. Showing veteran homeless breakdown as demographic analysis.')
    df2 = df.dropna(subset=['homeless_rate_per_10k', 'veteran_homeless'])
    df2['veteran_share_pct'] = (df2['veteran_homeless'] / df2['total_homeless'] * 100).round(1)
    fig = px.bar(
        df2.nlargest(20, 'veteran_share_pct').sort_values('veteran_share_pct'),
        x='veteran_share_pct', y='coc_name', orientation='h',
        color='homeless_rate_per_10k', color_continuous_scale='Reds',
        title='Veteran Share of Homeless Population by CoC (top 20)',
        labels={'veteran_share_pct': 'Veteran % of Homeless', 'coc_name': ''},
    )
    fig.show()

Census demographic data not available. Showing veteran homeless breakdown as demographic analysis.


## Interpretation

- **Demographic disparities**: Black Americans are overrepresented in homeless populations
- **Structural factors**: Systemic housing discrimination, wealth gaps, and institutional racism contribute
- **Veteran analysis**: Shows which CoCs have higher concentrations of veteran homelessness
- **Policy focus**: Targeted interventions for high-disparity CoCs